# 1. Análise e preparação do dataset
A equipe deverá conhecer e organizar os dados antes de iniciar a modelagem. Essa etapa deverá contemplar:

- Identificação das classes e da quantidade de imagens por classe.
- Verificação de imagens corrompidas, duplicadas ou com problemas de rotulagem.
- Análise do balanceamento entre as classes.
- Apresentação de exemplos das imagens e das principais dificuldades do problema.
- Divisão dos dados em conjuntos de treinamento e teste, com justificativa da estratégia utilizada.
  -Imagens do mesmo objeto, indivíduo ou sequência, quando existentes, deverão permanecer no mesmo conjunto para evitar vazamento de dados.

In [62]:
# Importações 
from pathlib import Path
from collections import Counter
import pandas as pd
import cv2
import hashlib

## 1.1 - Identificação das classes e da quantidade de imagens por classe.

As pastas train e test já estão separadas. Nelas há subpastas que representam cada classe. Dessa forma, cada diretório representa uma classe. ex: todas as imagens presentes em Apple Braeburn pertencem à mesma classe. 
O nome da pasta funciona como rótulo (label).

In [23]:
print(Path.cwd())
caminho_dataset = Path(r"fruits-360-100x100")
caminho_treino = caminho_dataset / "Training"
caminho_teste = caminho_dataset / "Test"

print(caminho_dataset.exists())
print(caminho_treino.exists())
print(caminho_teste.exists())

c:\Users\Samsung\AV2
True
True
True


In [24]:
# ---- IDENTIFICAÇÃO DAS CLASSES ----

classes_treino = []
classes_teste = []

for pasta in caminho_treino.iterdir():
    if pasta.is_dir():
        print(pasta.name)
        classes_treino.append(pasta.name)

for pasta in caminho_teste.iterdir():
    if pasta.is_dir():
        #print(pasta.name)
        classes_teste.append(pasta.name)

comparacao = set(classes_treino).difference(set(classes_teste))
comparacao_inversa = set(classes_teste).difference(set(classes_treino))


print(f"Quantidade de classes em treino: {len(classes_treino)}")
print(f"\nQuantidade de classes em teste: {len(classes_teste)}")
print(f"\nClasses faltantes no conjunto de teste com base no treino:\n{comparacao}")
print(f"\nClasses faltantes no conjunto de treino com base no teste:\n{comparacao_inversa}")

Almonds 1
Apple 10
Apple 11
Apple 12
Apple 13
Apple 14
Apple 17
Apple 18
Apple 19
Apple 20
Apple 21
Apple 22
Apple 23
Apple 5
Apple 6
Apple 7
Apple 8
Apple 9
Apple Braeburn 1
Apple Crimson Snow 1
Apple Golden 1
Apple Golden 2
Apple Golden 3
Apple Granny Smith 1
Apple Pink Lady 1
Apple Red 1
Apple Red 2
Apple Red 3
Apple Red Delicious 1
Apple Red Yellow 1
Apple Red Yellow 2
Apricot 1
Avocado 1
Avocado 2
Avocado Black 1
Avocado Black 2
Avocado Green 1
Banana 1
Banana 2
Banana 3
Banana 4
Banana Lady Finger 1
Banana Red 1
Bean pod 1
Beetroot 1
Blackberry 1
Blackberry 2
BlackBerry 3
BlackBerry 4
Blackberry 5
Blueberry 1
Cabbage red 1
Cabbage white 1
Cactus fruit 1
Cactus fruit green 1
Cactus fruit red 1
Caju seed 1
Cantaloupe 1
Cantaloupe 2
Cantaloupe 3
Carambola 1
Carambola 2
Carambola 3
Carrot 1
Cauliflower 1
Celery 1
Cherimoya 1
Cherry 1
Cherry 2
Cherry 3
Cherry 4
Cherry 5
Cherry Rainier 1
Cherry Rainier 2
Cherry Rainier 3
Cherry Sour 1
Cherry Wax 1
Cherry Wax 2
Cherry Wax Black 1
Cherry

Podemos notar que há duas classes presentes nos conjuntos que diferem a escrita. Dessa forma, o python interpreta como classes diferentes.

Podemos concluir que não há classes ausentes. Mas sim, nomeclaturas divergentes para uma mesma classe.
Isso justifica a utilização do método casefold().

In [25]:
classes_treino_normalizadas = [classe.casefold() for classe in classes_treino]
classes_teste_normalizadas = [classe.casefold() for classe in classes_teste]

comparacao = set(classes_treino_normalizadas).difference(set(classes_teste_normalizadas))
comparacao_inversa = set(classes_teste_normalizadas).difference(set(classes_treino_normalizadas))

print(f"\nClasses faltantes no conjunto de teste com base no treino:\n{comparacao}")
print(f"\nClasses faltantes no conjunto de treino com base no teste:\n{comparacao_inversa}")


Classes faltantes no conjunto de teste com base no treino:
set()

Classes faltantes no conjunto de treino com base no teste:
set()


In [26]:
# ---- CONTAGEM DAS IMAGENS POR CLASSE ----

pasta = caminho_treino / "Apple Braeburn 1"
imagens = list(pasta.glob("*.jpg"))
print(pasta.exists())
print(f"Quantidade de imagens da classe Apple Braeburn 1:\n{len(imagens)}")

True
Quantidade de imagens da classe Apple Braeburn 1:
492


In [27]:
# ---- CONTAGEM DAS IMAGENS POR CLASSE ----
dados_treino = []
dados_teste = []

def contagem_imagens(caminho, dados):
    for pasta in caminho.iterdir():
        if pasta.is_dir():
            imagens = list(pasta.glob("*.jpg"))
            quantidade = len(imagens)
            registro = {"classe": pasta.name, 
                        "quantidade": quantidade}
            dados.append(registro)
    return dados

df_treino = pd.DataFrame(contagem_imagens(caminho_treino, dados_treino))
df_teste = pd.DataFrame(contagem_imagens(caminho_teste, dados_teste))
print(f"---- Contagem das imagens por classe do conjunto de treino----")
df_treino.head()

---- Contagem das imagens por classe do conjunto de treino----


,classe,quantidade
0,Almonds 1,232
1,Apple 10,699
2,Apple 11,430
3,Apple 12,466
4,Apple 13,699


In [28]:
print(f"---- Contagem das imagens por classe do conjunto de teste----\n")
df_teste.head()

---- Contagem das imagens por classe do conjunto de teste----



,classe,quantidade
0,Almonds 1,77
1,Apple 10,231
2,Apple 11,142
3,Apple 12,154
4,Apple 13,235


In [29]:
print(50*("="))
print("------- INORMAÇÕES DO CONJUNTO DE TREINO ---------")
print(50*("="))

print(f"\nTamanho do df:{df_treino.shape}")
print(f"\n====== Informações: ======")
df_treino.info()
print(f"\n====== Total de imagens: =======")
print(df_treino['quantidade'].sum())
print(f"\n====== Estatísticas: ======")
df_treino.describe()

------- INORMAÇÕES DO CONJUNTO DE TREINO ---------

Tamanho do df:(278, 2)

====== Informações: ======
<class 'pandas.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   classe      278 non-null    str  
 1   quantidade  278 non-null    int64
dtypes: int64(1), str(1)
memory usage: 4.5 KB

====== Total de imagens: =======
150179

====== Estatísticas: ======


,quantidade
count,278.000000
mean,540.212230
std,171.233184
min,144.000000
25%,456.000000
50%,490.000000
75%,699.000000
max,984.000000


In [30]:
print(50*("="))
print("------- INORMAÇÕES DO CONJUNTO DE TESTE ---------")
print(50*("="))

print(f"\nTamanho:{df_teste.shape}")
print(f"\n====== Informações: ======")
df_teste.info()
print(f"\n====== Total de imagens: =======")
print(df_teste['quantidade'].sum())
print(f"\n====== Estatísticas: ======")

------- INORMAÇÕES DO CONJUNTO DE TESTE ---------

Tamanho:(278, 2)

====== Informações: ======
<class 'pandas.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   classe      278 non-null    str  
 1   quantidade  278 non-null    int64
dtypes: int64(1), str(1)
memory usage: 4.5 KB

====== Total de imagens: =======
50022

====== Estatísticas: ======


In [31]:
# Analisando se há alguma classe sem registro
df_treino.loc[df_treino["quantidade"] == 0]
df_teste.loc[df_teste["quantidade"] == 0]

,classe,quantidade


### Conclusões do tópico 1.1

O conjunto de dados Fruits-360 foi organizado em dois diretórios principais: Training e Test, nos quais cada subpasta representa uma classe. O nome de cada diretório foi utilizado como rótulo das imagens correspondentes.

Por meio de scripts desenvolvidos em Python, utilizando a biblioteca pathlib, foi realizada a identificação das classes e a contagem dos arquivos de imagem em cada diretório. Posteriormente, os resultados foram organizados em DataFrames utilizando a biblioteca Pandas.

Foram identificadas 278 classes tanto no conjunto de treinamento quanto no conjunto de teste. A contagem resultou em 150.179 imagens para treinamento e 50.022 imagens para teste, totalizando 200.201 imagens, em concordância com os valores apresentados na documentação oficial do dataset.

Durante a comparação dos nomes das classes, foi identificada uma divergência de capitalização na nomenclatura "BlackBerry 4" e "Blackberry 4". Após a normalização dos textos para comparação, verificou-se que se tratava da mesma classe.

A análise descritiva também demonstrou variações na quantidade de imagens entre as classes, indicando que a distribuição não é perfeitamente uniforme.

## 1.2 Verificação da integridade dos dados

### 1. Objetivo da etapa
Investigar se essas imagens apresentam problemas que possam comprometer as análises e os experimentos posteriores.


Além disso, faremos uma comparação entre scikit-image e OpenCV, observando aspectos como:
- Capacidade de leitura das imagens.
- Quantidade de falhas de leitura.
- Tempo de processamento.
- Diferenças nos resultados obtidos.

**Imagens corrompidas:** São arquivos que apresentam algum problema estrutural e não podem ser interpretados corretamente como imagens.
Para essa verificação, utilizaremos a biblioteca Pillow (biblioteca para leitura, manipulação e processamento de imagens).

In [32]:
# Identificando possíveis erros ao carregar as imagens - OpenCV

def CarregarImagem_CV2(caminho):
    try:
        imagem_opencv = cv2.imread(str(caminho))

        if imagem_opencv is None:
            return False

        else:
            return True

    except Exception:
        return False

In [34]:
## Verificação do conjunto de treinamento
resultados_CV2_treino = []

for pasta in caminho_treino.iterdir():
    if pasta.is_dir():
        for caminho_imagem in pasta.glob("*.jpg"):
            resultado = CarregarImagem_CV2(caminho_imagem)
            registro = {"classe": pasta.name, 
                        "imagem": caminho_imagem.name,
                        "leitura_sucesso": resultado}
            resultados_CV2_treino.append(registro)


## Verificação do conjunto de teste
resultados_CV2_teste = []

for pasta in caminho_teste.iterdir():
    if pasta.is_dir():
        for caminho_imagem in pasta.glob("*.jpg"):
            resultado = CarregarImagem_CV2(caminho_imagem)
            registro = {"classe": pasta.name, 
                        "imagem": caminho_imagem.name, 
                        "leitura_sucesso": resultado}
            resultados_CV2_teste.append(registro)            



In [56]:
# Validando os resultados
print("="*50)
print(f"Imagens analisadas no treinamento: {len(resultados_CV2_treino)}")
print(f"Imagens analisadas no teste: {len(resultados_CV2_teste)}")
print(f"Resultados do treino: \n{resultados_CV2_treino[:5]}")
print(f"Resultados do teste: \n{resultados_CV2_teste[:5]}")
print("="*50)

Imagens analisadas no treinamento: 150179
Imagens analisadas no teste: 50022
Resultados do treino: 
[{'classe': 'Almonds 1', 'imagem': 'r0_0_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_100_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_101_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_102_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_104_100.jpg', 'leitura_sucesso': True}]
Resultados do teste: 
[{'classe': 'Almonds 1', 'imagem': 'r0_103_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_107_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_111_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_115_100.jpg', 'leitura_sucesso': True}, {'classe': 'Almonds 1', 'imagem': 'r0_119_100.jpg', 'leitura_sucesso': True}]


In [36]:
# Armazenando os resultados em dataframes
df_resultados_CV2_treino = pd.DataFrame(resultados_CV2_treino)
df_resultados_CV2_teste = pd.DataFrame(resultados_CV2_teste)

### Verificando a estrutura dos DataFrames


In [49]:
df_resultados_CV2_treino.head()
print("="*50)
print(f"Tamanho do df de treino: {df_resultados_CV2_treino.shape}")
print("="*50)
print(df_resultados_CV2_treino.info())
print("="*50)

Tamanho do df de treino: (150179, 3)
<class 'pandas.DataFrame'>
RangeIndex: 150179 entries, 0 to 150178
Data columns (total 3 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   classe           150179 non-null  str  
 1   imagem           150179 non-null  str  
 2   leitura_sucesso  150179 non-null  bool 
dtypes: bool(1), str(2)
memory usage: 2.4 MB
None


In [52]:
df_resultados_CV2_teste.head()
print("="*50)
print(f"Tamanho do df de teste: {df_resultados_CV2_teste.shape}")
print("="*50)
print(df_resultados_CV2_teste.info())
print("="*50)

Tamanho do df de teste: (50022, 3)
<class 'pandas.DataFrame'>
RangeIndex: 50022 entries, 0 to 50021
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   classe           50022 non-null  str  
 1   imagem           50022 non-null  str  
 2   leitura_sucesso  50022 non-null  bool 
dtypes: bool(1), str(2)
memory usage: 830.6 KB
None


### Contabilizando as imagens carregadas corretamente

In [53]:
# Imagens carregadas corretementeno treinamento
print("="*50)
sucesso_treino = df_resultados_CV2_treino["leitura_sucesso"].sum()
print(f"Imagens carregadas corretemente no treinamento: {sucesso_treino}")

# Imagens carregadas corretementeno teste
sucesso_teste = df_resultados_CV2_teste["leitura_sucesso"].sum()
print(f"Imagens carregadas corretement no teste: {sucesso_teste}")
print("="*50)

Imagens carregadas corretemente no treinamento: 150179
Imagens carregadas corretement no teste: 50022


### Contabilizando as imagens carregadas incorretamente

In [54]:
# Imagens carregadas incorretementeno treino
print("="*50)
falhas_treino = (df_resultados_CV2_treino["leitura_sucesso"] == False).sum()
print(f"Falhas de leitura no treinamento: {falhas_treino}")

# Imagens carregadas incorretementeno teste
falhas_teste = (df_resultados_CV2_teste["leitura_sucesso"] == False).sum()
print(f"Falhas de leitura no teste: {falhas_teste}")
print("="*50)

Falhas de leitura no treinamento: 0
Falhas de leitura no teste: 0


### Calculando a taxa de sucesso de leitura

Taxa de sucesso = Imagens carregadas corretamente / Total de imagens analidadas * 100

In [57]:
# calculando a taxa de sucesso - treino
print("="*50)
taxa_sucesso_treino = (sucesso_treino / len(df_resultados_CV2_treino)) * 100
print(f"Taxa de sucesso no treinamento: {taxa_sucesso_treino:.2f}%")

# teste
taxa_sucesso_teste = (sucesso_teste / len(df_resultados_CV2_teste)) * 100
print(f"Taxa de sucesso no teste: {taxa_sucesso_teste:.2f}%")
print("="*50)

Taxa de sucesso no treinamento: 100.00%
Taxa de sucesso no teste: 100.00%


### Verificando quais imagens apresentaram falhas

In [61]:
# verificando falhas - treino
imagens_falhas_treino = df_resultados_CV2_treino.loc[df_resultados_CV2_treino["leitura_sucesso"] == False]
imagens_falhas_treino

# teste
imagens_falhas_teste = df_resultados_CV2_teste.loc[df_resultados_CV2_teste["leitura_sucesso"] == False]
imagens_falhas_teste

,classe,imagem,leitura_sucesso


## 1.3 - Verificação de imagens duplicadas

**1. Objetivo** \
Precisamos investigar se existem imagens duplicadas no dataset.

Uma duplicata ocorre quando duas ou mais imagens representam exatamente o mesmo conteúdo visual, mesmo que estejam armazenadas em arquivos diferentes.

**2. Existem diferentes tipos de duplicatas** \
Antes de implementar, precisamos entender que não existe apenas uma maneira de definir uma imagem duplicada.

**2.1. Duplicatas exatas** \
São arquivos cujos pixels são exatamente iguais. Para detectar esse tipo de duplicata, podemos utilizar hashes.

**2.2. Duplicatas visuais ou semelhantes** \
São imagens visualmente semelhantes, mas não necessariamente idênticas. Por exemplo: A mesma fruta em uma posição ligeiramente diferente.

Para isso, existem métodos como:
- Hash perceptual (Perceptual Hash).
- Similaridade estrutural (SSIM).
- Comparação baseada em características visuais.

**3. Primeira abordagem: hash das imagens** \
3.1. Um hash é uma representação calculada a partir dos dados de um arquivo.

Podemos pensar nele como uma espécie de assinatura. Se dois arquivos possuem exatamente o mesmo conteúdo, normalmente terão o mesmo hash utilizando um algoritmo determinístico.

Exemplo conceitual:\
Arquivo	: imagem_1.jpg                
Hash : A72F91...

Para essa análise, podemos utilizar a biblioteca padrão do Python chamada hashlib (biblioteca nativa do Python que fornece funções para calcular hashes.).

Vamos utilizar o algoritmo SHA-256.

Esse algoritmo produz uma assinatura de 256 bits, normalmente representada por uma sequência hexadecimal de 64 caracteres.

In [63]:
# calculando hash
def Calcula_hash(caminho_imagem):
        imagem_binary = caminho_imagem.read_bytes()     # lê o conteúdo de um arquivo e retorna seus dados em binário (bytes).
        obj_hash = hashlib.sha256()                     # Criamos o objeto vazio.         
        obj_hash.update(imagem_binary)                  # Fornecemos os bytes por meio de update().
        hash_hexa = obj_hash.hexdigest()                # Converte o resultado do hash em uma representação hexadecimal.
        return hash_hexa

In [ ]:
# Teste para duas imagens
# pasta = caminho_treino / "Almonds 1"
# imagem_1 = pasta / "r0_0_100.jpg"
# imagem_2 = pasta / "r0_1_100.jpg"
# hash_1 = Calcula_hash(imagem_1)
# hash_2 = Calcula_hash(imagem_2)
# print(hash_1)
# print(hash_2)


027b4933da878841c729fc83f81456dfee2ab3d7a54bbeea7433c2e94ea5cae5
34918d880a263187928808776440767859389fa737565d3dd3e79a2e8d9ab5ae


In [70]:
# Calculando os hashes das imagens de treinamento
hashes_treino = []
for pasta in caminho_treino.iterdir():
    if pasta.is_dir():
        for caminho_imagem in pasta.glob("*.jpg"):
            hash_imagem = Calcula_hash(caminho_imagem)
            registro = {"classe": pasta.name,
                        "imagem": caminho_imagem.name,
                        "hash": hash_imagem,
                        "conjunto": "Training"}
            hashes_treino.append(registro)

In [71]:
# Calculando os hashes das imagens de teste
hashes_teste = []
for pasta in caminho_teste.iterdir():
    if pasta.is_dir():
        for caminho_imagem in pasta.glob("*.jpg"):
            hash_imagem = Calcula_hash(caminho_imagem)
            registro = {"classe": pasta.name,
                        "imagem": caminho_imagem.name,
                        "hash": hash_imagem,
                        "conjunto": "Test"}
            hashes_teste.append(registro)

In [72]:
# Transformando os resultados em DataFrames
df_hashes_treino = pd.DataFrame(hashes_treino)
df_hashes_teste = pd.DataFrame(hashes_teste)

print("Imagens analisadas no treinamento:", len(df_hashes_treino))
print("Imagens analisadas no teste:", len(df_hashes_teste))

Imagens analisadas no treinamento: 150179
Imagens analisadas no teste: 50022


In [73]:
# Visualizando os registros
print("Visualizando os registros")
display(df_hashes_treino.head())
display(df_hashes_teste.head())

print("Verificando a estrutura")
df_hashes_treino.info()
df_hashes_teste.info()

Visualizando os registros


,classe,imagem,hash,conjunto
0,Almonds 1,r0_0_100.jpg,027b4933da878841c729fc83f81456dfee2ab3d7a54bbe...,Training
1,Almonds 1,r0_100_100.jpg,0bb2a25f042288ac65ff87cc8051ec791a42ede3b062bb...,Training
2,Almonds 1,r0_101_100.jpg,9c9d4049314cdb9a0e2e0942f8017fdcbfb32f6ef81813...,Training
3,Almonds 1,r0_102_100.jpg,ae65bd696620e0564f7a09652f63698679947095fda08e...,Training
4,Almonds 1,r0_104_100.jpg,9adc4e597b2747d44b745bf57c760aa2322ccfd6418672...,Training


,classe,imagem,hash,conjunto
0,Almonds 1,r0_103_100.jpg,cfd4d325fb5cb76a9d90212bb9b61a8da999ec781abe20...,Test
1,Almonds 1,r0_107_100.jpg,3c2f317c78e6f6fbfda665c0be3b8cf1ac9ce3b5e27ada...,Test
2,Almonds 1,r0_111_100.jpg,7b868c085a3b1870b53b53401e9f1f7e42938e3d7abf92...,Test
3,Almonds 1,r0_115_100.jpg,e952078e373a7d0c21c8f7eeefd8bfbbc2027dbca02153...,Test
4,Almonds 1,r0_119_100.jpg,f1d40bc37ea4e1ae9ad425156c5eef63acbfaae4b5df2c...,Test


Verificando a estrutura
<class 'pandas.DataFrame'>
RangeIndex: 150179 entries, 0 to 150178
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   classe    150179 non-null  str  
 1   imagem    150179 non-null  str  
 2   hash      150179 non-null  str  
 3   conjunto  150179 non-null  str  
dtypes: str(4)
memory usage: 4.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 50022 entries, 0 to 50021
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   classe    50022 non-null  str  
 1   imagem    50022 non-null  str  
 2   hash      50022 non-null  str  
 3   conjunto  50022 non-null  str  
dtypes: str(4)
memory usage: 1.5 MB


In [74]:
# Verificando duplicatas - treino
duplicatas_treino = df_hashes_treino.duplicated(subset="hash")
print("Quantidade de duplicatas no treinamento:", duplicatas_treino.sum())

df_duplicatas_treino = df_hashes_treino.loc[
    df_hashes_treino.duplicated(subset="hash", keep=False)
]

display(df_duplicatas_treino)

Quantidade de duplicatas no treinamento: 3


,classe,imagem,hash,conjunto
17968,Avocado 2,r_100.jpg,4ffc5040e8335c3f226cb8fd4904bddeffa7245c322ce0...,Training
18210,Avocado 2,r_7_100.jpg,4ffc5040e8335c3f226cb8fd4904bddeffa7245c322ce0...,Training
41522,Cherry Wax Red 1,0_100.jpg,8c97b2e04d85a7edb567b1b612579c733ab66e4c0042f8...,Training
41609,Cherry Wax Red 1,1_100.jpg,8c97b2e04d85a7edb567b1b612579c733ab66e4c0042f8...,Training
106200,Pear Forelle 1,r_226_100.jpg,4fb17a31635bbaf766f2705d7f440fa6e7f2f9dd8fdac7...,Training
106201,Pear Forelle 1,r_227_100.jpg,4fb17a31635bbaf766f2705d7f440fa6e7f2f9dd8fdac7...,Training


In [75]:
# Verificando duplicatas - teste
duplicatas_teste = df_hashes_teste.duplicated(subset="hash")
print("Quantidade de duplicatas no teste:", duplicatas_teste.sum())

df_duplicatas_teste = df_hashes_teste.loc[
    df_hashes_teste.duplicated(subset="hash", keep=False)
]

display(df_duplicatas_teste)

Quantidade de duplicatas no teste: 0


,classe,imagem,hash,conjunto


In [76]:
# Identificando duplicatas entre treinamento e teste
hashes_unicos_treino = set(df_hashes_treino["hash"])
hashes_unicos_teste = set(df_hashes_teste["hash"])

hashes_comuns = hashes_unicos_treino.intersection(hashes_unicos_teste)
print("Quantidade de hashes presentes em ambos os conjuntos:", len(hashes_comuns))

df_duplicatas_entre_conjuntos = pd.concat(
    [df_hashes_treino, df_hashes_teste]
)

df_duplicatas_entre_conjuntos = df_duplicatas_entre_conjuntos.loc[
    df_duplicatas_entre_conjuntos["hash"].isin(hashes_comuns)
]

display(df_duplicatas_entre_conjuntos)

Quantidade de hashes presentes em ambos os conjuntos: 0


,classe,imagem,hash,conjunto


### Conclusões do tópico 1.3

A análise de duplicatas exatas utilizando hashes SHA-256 identificou três pares de imagens duplicadas no conjunto de treinamento, totalizando seis arquivos envolvidos. Não foram encontradas duplicatas no conjunto de teste nem imagens idênticas compartilhadas entre treinamento e teste. Como as duplicatas identificadas estão exclusivamente no treinamento e representam uma quantidade insignificante em relação ao tamanho total do conjunto, optou-se inicialmente por manter os arquivos originais.